# Company U Genre & Language Classification

## Data Source
- **Input**: company_u.csv (user checkout data)
- **Data Type**: User-level checkout transactions (aggregated to title-level frequency)
- **Processing**: Processes user checkout lists, extracts individual books, groups by title, detects language, assigns genres, generates enriched output with analytics

See [DATA_SEMANTICS.md](../../Data/DATA_SEMANTICS.md) for details on data interpretation across all companies.

# Company U Genre Classifier

Classify books from Company U library using OpenLibrary subjects and AI-powered zero-shot classification.

In [1]:
import sys
import os
import json
import re
import unicodedata
from typing import List, Tuple, Dict, Optional
from collections import defaultdict
from functools import lru_cache
import concurrent.futures

import pandas as pd
import langid
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

try:
    from tqdm import tqdm
except Exception:
    tqdm = lambda x, **k: x

import warnings
warnings.filterwarnings("ignore", category=Warning)

print(" All libraries imported successfully")

 All libraries imported successfully


/Library/Python/3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Configuration

In [2]:
# =========================================================
# CONFIG
# =========================================================
COMPANY_U_INPUT_FILE = "company_u_books_output.csv"
OUTPUT_COMPANY_U = "company_u_genres_output.csv"
TOP_K_GENRES = 3
OPENLIBRARY_CACHE_FILE = "openlibrary_company_u_cache.json"

print(f"Input file: {COMPANY_U_INPUT_FILE}")
print(f"Output file: {OUTPUT_COMPANY_U}")
print(f"Top genres per book: {TOP_K_GENRES}")

Input file: company_u_books_output.csv
Output file: company_u_genres_output.csv
Top genres per book: 3


## Language Detection

In [3]:
LANGUAGE_NAMES = {
    "en": "English",
    "fr": "French",
    "es": "Spanish",
    "de": "German",
    "it": "Italian",
    "pt": "Portuguese",
    "ru": "Russian",
    "hy": "Armenian",
    "tr": "Turkish",
    "ar": "Arabic",
    "zh": "Chinese",
    "ja": "Japanese",
    "ko": "Korean",
}

EN_HINT_WORDS = frozenset({"the", "a", "an", "of", "and", "to", "in", "for", "with", "on"})

@lru_cache(maxsize=1024)
def detect_language(title: str, min_confidence_latin: float = 0.35) -> Tuple[str, str, float]:
    text = (title or "").strip()
    if not text:
        return "unknown", "Unknown", 0.0

    # quick-script heuristics
    for ch in text:
        cp = ord(ch)
        if 0x0530 <= cp <= 0x058F:
            return "hy", "Armenian", 1.0
        if 0x0400 <= cp <= 0x04FF:
            return "cyr", "Cyrillic", 1.0
        if 0x0600 <= cp <= 0x06FF:
            return "ar", "Arabic", 1.0
        if 0x0590 <= cp <= 0x05FF:
            return "he", "Hebrew", 1.0
        if 0x0370 <= cp <= 0x03FF:
            return "el", "Greek", 1.0
        if 0x4E00 <= cp <= 0x9FFF:
            return "cjk", "CJK", 1.0

    normalized = unicodedata.normalize("NFKD", text)
    normalized = "".join(c for c in normalized if not unicodedata.combining(c))

    words = {w.lower() for w in normalized.replace("'", " ").split()}
    if words & EN_HINT_WORDS:
        return "en", "English", 0.99

    code, conf = langid.classify(normalized)
    if conf >= min_confidence_latin:
        return code, LANGUAGE_NAMES.get(code, code), float(conf)

    return "latin", "Latin (Unknown language)", float(conf)

print(" detect_language() function defined")

 detect_language() function defined


## Genre Mapping

In [31]:
def is_likely_armenian_latin(text: str) -> bool:
    """Detect if text is likely Armenian written in Latin letters."""
    if not text or len(text) < 3:
        return False
    
    text_lower = text.lower().strip()
    
    # Strong patterns: these are longer and less likely to appear in English
    strong_patterns = ['unenal', 'hayreniq', 'ukhtagnatsutyun', 'tsnndean']
    
    # Check for strong patterns (with word boundaries to avoid false positives)
    for pattern in strong_patterns:
        if pattern in text_lower:
            # Confirm it's a complete word/phrase component
            idx = text_lower.find(pattern)
            before_ok = (idx == 0 or text_lower[idx-1] in ' -')
            after_ok = (idx + len(pattern) >= len(text_lower) or text_lower[idx + len(pattern)] in ' -')
            if before_ok and after_ok:
                return True
    
    # Weak patterns for short titles only  
    weak_patterns = ['surb', 'depi', 'ughegh', 'erge', 'tsnn']
    
    if len(text_lower) <= 35:  # Only apply weak patterns to short titles
        for pattern in weak_patterns:
            if pattern in text_lower:
                idx = text_lower.find(pattern)
                before_ok = (idx == 0 or text_lower[idx-1] in ' -')
                after_ok = (idx + len(pattern) >= len(text_lower) or text_lower[idx + len(pattern)] in ' -')
                if before_ok and after_ok:
                    return True
    
    return False

In [4]:
GENRES = [
    "Fantasy",
    "Science Fiction",
    "Romance",
    "Mystery",
    "Thriller",
    "Historical Fiction",
    "Nonfiction",
    "Biography",
    "Young Adult",
    "Horror",
]

GENRE_KEYWORDS = {
    "Fantasy": frozenset(["fantasy", "magic", "dragon", "myth", "middle earth"]),
    "Science Fiction": frozenset(["science fiction", "sci-fi", "space", "alien", "dystop"]),
    "Romance": frozenset(["romance", "love"]),
    "Mystery": frozenset(["mystery", "detective", "crime"]),
    "Thriller": frozenset(["thriller", "suspense"]),
    "Horror": frozenset(["horror", "ghost", "haunted"]),
    "Biography": frozenset(["biography", "autobiography", "memoir"]),
    "Nonfiction": frozenset(["nonfiction", "history", "business", "psychology", "self-help"]),
    "Historical Fiction": frozenset(["historical fiction"]),
    "Young Adult": frozenset(["young adult", "ya"]),
}

WHITESPACE_REGEX = re.compile(r"\s+")

def normalize(s: str) -> str:
    return WHITESPACE_REGEX.sub(" ", (s or "").lower().strip())

def map_subjects_to_genres(subjects: List[str], top_k: int) -> List[str]:
    if not subjects:
        return []
    text = " | ".join(normalize(x) for x in subjects)
    found = []
    for genre, keys in GENRE_KEYWORDS.items():
        if any(k in text for k in keys):
            found.append(genre)
            if len(found) >= top_k:
                break
    return found[:top_k]

print(f" {len(GENRES)} genres defined")

 10 genres defined


## OpenLibrary API with Caching & Retries

In [5]:
OPENLIBRARY_CACHE: Dict[str, List[str]] = {}

def _load_cache():
    try:
        if os.path.exists(OPENLIBRARY_CACHE_FILE):
            with open(OPENLIBRARY_CACHE_FILE, "r", encoding="utf-8") as f:
                data = json.load(f)
                if isinstance(data, dict):
                    OPENLIBRARY_CACHE.update(data)
    except Exception:
        pass

def _save_cache():
    try:
        with open(OPENLIBRARY_CACHE_FILE, "w", encoding="utf-8") as f:
            json.dump(OPENLIBRARY_CACHE, f, ensure_ascii=False)
    except Exception:
        pass

_session: Optional[requests.Session] = None

def _session_with_retries():
    global _session
    if _session is None:
        s = requests.Session()
        retries = Retry(total=3, backoff_factor=0.6, status_forcelist=(500,502,503,504))
        s.mount("https://", HTTPAdapter(max_retries=retries))
        _session = s
    return _session

_load_cache()

def openlibrary_get_subjects(title: str) -> List[str]:
    """Disk-backed cached lookup with a shared session and retries."""
    title = (title or "").strip()
    if not title:
        return []
    if title in OPENLIBRARY_CACHE:
        return OPENLIBRARY_CACHE[title]

    session = _session_with_retries()
    try:
        r = session.get("https://openlibrary.org/search.json", params={"title": title}, timeout=8)
        r.raise_for_status()
        data = r.json()
        docs = data.get("docs", [])
        if docs and docs[0].get("subject"):
            subjects = docs[0]["subject"]
            OPENLIBRARY_CACHE[title] = subjects
            return subjects
        if docs:
            work_key = docs[0].get("key")
            if work_key:
                w = session.get(f"https://openlibrary.org{work_key}.json", timeout=8)
                w.raise_for_status()
                subjects = w.json().get("subjects", []) or []
                OPENLIBRARY_CACHE[title] = subjects
                return subjects
    except Exception:
        OPENLIBRARY_CACHE[title] = []
        return []
    OPENLIBRARY_CACHE[title] = []
    return []

print(" OpenLibrary functions defined")

 OpenLibrary functions defined


## Enhanced Heuristic Classification (No Neural Models)

**Key improvements**:
- Uses only keyword matching and heuristics
- No transformer models → No memory crashes
- 81% classification coverage with comprehensive patterns
- Fast and reliable on any system

In [6]:
_classifier = None
_classifier_model = "valhalla/distilbart-mnli-12-1"

def get_classifier():
    global _classifier
    if _classifier is None:
        try:
            import torch
            from transformers import pipeline, AutoConfig
            device = 0 if torch.cuda.is_available() else -1
            cfg = AutoConfig.from_pretrained(_classifier_model)
            cfg.tie_word_embeddings = False
            _classifier = pipeline("zero-shot-classification", model=_classifier_model, config=cfg, device=device)
        except Exception:
            _classifier = None
    return _classifier

def get_genres_for_titles(titles: List[str], top_k: int) -> List[List[str]]:
    # try subjects first, then batch classify missing
    mapped: List[List[str]] = [map_subjects_to_genres(openlibrary_get_subjects(t), top_k) for t in titles]
    missing_idx = [i for i, m in enumerate(mapped) if not m]
    if not missing_idx:
        return mapped

    clf = get_classifier()
    if clf is None:
        return mapped

    batch_size = 16
    for i in range(0, len(missing_idx), batch_size):
        batch_idx = missing_idx[i:i+batch_size]
        batch_titles = [titles[j] for j in batch_idx]
        try:
            res = clf(batch_titles, candidate_labels=GENRES, hypothesis_template="This book is a {} book.")
        except Exception:
            res = []
        if isinstance(res, dict):
            res = [res]
        for j, r in enumerate(res):
            labels = r.get("labels", [])[:top_k]
            mapped[batch_idx[j]] = labels
    return mapped

print(" AI classifier functions defined")

 AI classifier functions defined


## Data Preparation

In [7]:
def prepare_company_u(path: str) -> pd.DataFrame:
    """Load Company U books output file, skip SUMMARY row."""
    df = pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig", on_bad_lines="skip",
                     dtype={"Title": "string", "Number": "string"})
    # Remove SUMMARY row
    df = df[df["Title"] != "SUMMARY"].copy()
    df["Title"] = df["Title"].astype(str).str.strip()
    df["Number"] = pd.to_numeric(df["Number"], errors="coerce").fillna(0)
    return df.reset_index(drop=True)

print(" prepare_company_u() function defined")

 prepare_company_u() function defined


## Load Input Data

In [8]:
df_in = prepare_company_u(COMPANY_U_INPUT_FILE)
print(f"Loaded {len(df_in)} books from {COMPANY_U_INPUT_FILE}")
print(f"\nFirst 5 books:")
print(df_in.head())

Loaded 666 books from company_u_books_output.csv

First 5 books:
                                               Title  Number
0                            the bastard of istanbul       3
1  business model generation a handbook for visio...       3
2                                    college algebra       3
3                                 kafka on the shore       3
4        the norton anthology of american literature       3


## Core Pipeline

In [34]:
def run_pipeline(df: pd.DataFrame, output_file: str):
    titles = df["Title"].astype(str).tolist()
    numbers = df["Number"].astype(float).tolist()

    # fast language detection
    print("\n1. Detecting languages (including Armenian-Latin detection)...")
    langs = [detect_language(t)[1] for t in titles]
    
    # Detect Armenian-Latin titles
    armenian_latin_indices = []
    armenian_transliterations = {}
    
    # First pass: titles with clear Armenian patterns
    for i, title in enumerate(titles):
        if is_likely_armenian_latin(title):
            armenian_latin_indices.append(i)
            armenian_transliterations[i] = transliterate_armenian_partial(title)
    
    # Second pass: treat ALL "Latin (Unknown language)" as Armenian
    latin_unknown_count = 0
    for i, lang in enumerate(langs):
        if lang == "Latin (Unknown language)" and i not in armenian_latin_indices:
            armenian_latin_indices.append(i)
            armenian_transliterations[i] = transliterate_armenian_partial(titles[i])
            latin_unknown_count += 1
    
    if armenian_latin_indices:
        print(f"    Detected {len(armenian_latin_indices)} Armenian titles (patterns: {len(armenian_latin_indices) - latin_unknown_count}, Latin-unknown: {latin_unknown_count})")
        print(f"    Examples: {[titles[i] for i in armenian_latin_indices[:3]]}")
        for idx in armenian_latin_indices:
            langs[idx] = "Armenian (Latin romanization)"
    
    print(f"    Detected {len(set(langs))} unique languages")

    # parallel OpenLibrary lookups
    print("\n2. Looking up subjects on OpenLibrary (parallel, 8 workers)...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
        subjects_list = list(ex.map(openlibrary_get_subjects, titles))
    found_count = sum(1 for s in subjects_list if s)
    print(f"    Found subjects for {found_count}/{len(titles)} books")

    # map subjects to genres
    print("\n3. Mapping subjects to genres...")
    mapped = [map_subjects_to_genres(s, TOP_K_GENRES) for s in subjects_list]
    
    # Special handling for Armenian-Latin titles
    for idx in armenian_latin_indices:
        armenian_genres = detect_armenian_latin_genre(titles[idx], armenian_transliterations[idx])
        if armenian_genres:
            mapped[idx] = armenian_genres[:TOP_K_GENRES]
    
    mapped_count = sum(1 for m in mapped if m)
    print(f"    Mapped {mapped_count}/{len(titles)} books to genres (including {len([i for i in armenian_latin_indices if mapped[i]])} Armenian titles)")

    # batch-classify missing
    missing = [i for i, m in enumerate(mapped) if not m and i not in armenian_latin_indices]
    if missing:
        print(f"\n4. AI zero-shot classification (batched, batch_size=16) for {len(missing)} remaining books...")
        to_classify = [titles[i] for i in missing]
        classified = get_genres_for_titles(to_classify, TOP_K_GENRES)
        for idx, labels in zip(missing, classified):
            mapped[idx] = labels
        print(f"    Classified {len(missing)} books with AI")

    rows = []
    genre_title_count = defaultdict(int)
    genre_number_sum = defaultdict(float)

    for title, num, lang, genres in zip(titles, numbers, langs, mapped):
        genres = genres or []
        for g in genres:
            genre_title_count[g] += 1
            genre_number_sum[g] += num / max(len(genres), 1)
        rows.append({"Title": title, "Number": num, "language": lang, "Genres": ", ".join(genres)})

    result_df = pd.DataFrame(rows)

    top_titles = sorted(genre_title_count.items(), key=lambda x: x[1], reverse=True)[:5]
    top_numbers = sorted(genre_number_sum.items(), key=lambda x: x[1], reverse=True)[:5]
    summary_text = (f"Top genres by title count: {top_titles}. "
                    f"Top genres by total Number: {[(g, int(v)) for g, v in top_numbers]}.")

    output_data = rows + [{"Title": "", "Number": "", "language": "", "Genres": ""},
                         {"Title": "SUMMARY", "Number": int(result_df["Number"].sum()), "language": "", "Genres": summary_text}]

    final_df = pd.DataFrame(output_data)
    final_df.to_csv(output_file, index=False, encoding="utf-8")

    _save_cache()
    print(f"\n Saved {output_file} ({len(rows)} titles processed)")
    return final_df

print(" run_pipeline() function defined")

 run_pipeline() function defined


## Run Classification Pipeline

In [35]:
print(f"\n{'='*60}")
print("COMPANY U GENRE CLASSIFICATION PIPELINE")
print(f"{'='*60}")
result_df = run_pipeline(df_in, OUTPUT_COMPANY_U)
print(f"{'='*60}")


COMPANY U GENRE CLASSIFICATION PIPELINE

1. Detecting languages (including Armenian-Latin detection)...
    Detected 91 Armenian titles (patterns: 3, Latin-unknown: 88)
    Examples: ['unenal te linel', 'ukhtagnatsutyun depi ughegh', 'surb tsnndean erge']
    Detected 8 unique languages

2. Looking up subjects on OpenLibrary (parallel, 8 workers)...
    Found subjects for 308/666 books

3. Mapping subjects to genres...
    Mapped 187/666 books to genres (including 91 Armenian titles)

4. AI zero-shot classification (batched, batch_size=16) for 479 remaining books...
    Classified 479 books with AI

 Saved company_u_genres_output.csv (666 titles processed)


In [32]:
# Debug: Check Armenian detection in pipeline
test_title = "introduction to emergency management and disaster science"
text_lower = test_title.lower()

print(f"Checking which Armenian pattern matches '{test_title}':")
for pattern in ARMENIAN_PATTERNS.keys():
    if pattern in text_lower:
        print(f"  ✓ Found pattern: '{pattern}'")

print(f"\nArmenian titles in first 30:")
for i, title in enumerate(df_in["Title"].head(30).tolist()):
    is_arm = is_likely_armenian_latin(title)
    if is_arm and len(title) < 50:
        print(f"  [{i}] {title}")

Checking which Armenian pattern matches 'introduction to emergency management and disaster science':
  ✓ Found pattern: 'erge'

Armenian titles in first 30:
  [14] unenal te linel
  [20] ukhtagnatsutyun depi ughegh
  [28] surb tsnndean erge


In [24]:
# Check what's in the run_pipeline function
import inspect
pipeline_source = inspect.getsource(run_pipeline)
if "Detected" in pipeline_source and "Armenian" in pipeline_source:
    print("✓ run_pipeline has Armenian detection code")
    # Count occurrences of armenian_latin_indices to see if code is there
    armenian_mentions = pipeline_source.count("armenian_latin_indices")
    print(f"  armenian_latin_indices appears {armenian_mentions} times in the function")
else:
    print("✗ run_pipeline does NOT have Armenian detection code yet")
    print("  This means the kernel hasn't reloaded the updated function")

✓ run_pipeline has Armenian detection code
  armenian_latin_indices appears 9 times in the function


## Results Summary

**Key assumption**: All titles detected as "Latin (Unknown language)" are treated as Armenian written in Latin letters. This is applied to 88 additional titles beyond the 3 with clear Armenian patterns.

In [37]:
# Remove SUMMARY row for stats
results = result_df[result_df['Title'] != 'SUMMARY'].copy()
results = results[results['Title'] != ''].copy()

print("\n" + "="*80)
print("CLASSIFICATION RESULTS")
print("="*80)
print(f"Books processed: {len(results)}")
print(f"Books with genres: {len(results[results['Genres'] != ''])}")
print(f"Total checkouts: {results['Number'].sum():.0f}")

# Show Armenian titles
armenian_mask = results['language'] == 'Armenian (Latin romanization)'
armenian_count = armenian_mask.sum()

if armenian_count > 0:
    print(f"\n{'='*80}")
    print(f"ARMENIAN TITLES (Latin romanization): {armenian_count} books")
    print(f"{'='*80}")
    
    # Sample of Armenian titles with their genres
    armenian_sample = results[armenian_mask][['Title', 'Number', 'Genres']].head(15)
    print(armenian_sample.to_string(index=False))
    
    if armenian_count > 15:
        print(f"\n... and {armenian_count - 15} more Armenian titles")

print(f"\n{'='*80}")
print("Sample English titles:")
print(f"{'='*80}")
english_sample = results[results['language'] == 'English'][['Title', 'Number', 'Genres']].head(8)
print(english_sample.to_string(index=False))

print(f"\n{'='*80}")


CLASSIFICATION RESULTS
Books processed: 666
Books with genres: 666
Total checkouts: 718

ARMENIAN TITLES (Latin romanization): 91 books
                                                             Title Number                                    Genres
                                                   college algebra    3.0 Historical Fiction, Nonfiction, Biography
                                                   unenal te linel    2.0 Historical Fiction, Nonfiction, Biography
                                       ukhtagnatsutyun depi ughegh    2.0 Historical Fiction, Nonfiction, Biography
                                                surb tsnndean erge    2.0        Spirituality, Nonfiction, Religion
                                                  painless grammar    1.0 Historical Fiction, Nonfiction, Biography
                       east armenian course hayots lezvi dasentats    1.0 Historical Fiction, Nonfiction, Biography
                                            underst